# SOFR Butterfly Relative Value Screener

Three-layer signal framework for evaluating every 3-month butterfly across the SR3 strip:

1. **Variance Consistency** — BKM model-free moments (integration, not BL differentiation)
2. **Probability Space** — Kink sharpness vs distributional uncertainty
3. **Fragility** — Tail sensitivity of the weakest leg

Plus FOMC meeting overlay for structural curvature justification.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../../../")

import datetime
import math
import warnings

import numpy as np
import pandas as pd
import QuantLib as ql
import pytz

NYC_tz = pytz.timezone("America/New_York")

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.stir_curve_building_utils import get_fomc_meetings_list
from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from RVUtils.ImpliedDistribution import SFRImpliedDistribution

In [2]:
# ─── Configuration ────────────────────────────────────────────────────────────
AS_OF = datetime.date(2026, 5, 22)
CURVE = "USD-SOFR-1D-Q12STIRT"

IMM_TENORS = [
    "IMM_1xIMM_2", "IMM_2xIMM_3", "IMM_3xIMM_4", "IMM_4xIMM_5",
    "IMM_5xIMM_6", "IMM_6xIMM_7", "IMM_7xIMM_8", "IMM_8xIMM_9",
    "IMM_9xIMM_10", "IMM_10xIMM_11", "IMM_11xIMM_12", "IMM_12xIMM_13",
]

IMM_CODE_TO_SFR = {
    "M6": "SFRM26", "U6": "SFRU26", "Z6": "SFRZ26",
    "H7": "SFRH27", "M7": "SFRM27", "U7": "SFRU27", "Z7": "SFRZ27",
    "H8": "SFRH28", "M8": "SFRM28", "U8": "SFRU28", "Z8": "SFRZ28",
    "H9": "SFRH29",
}
IMM_MONTH_MAP = {"H": 3, "M": 6, "U": 9, "Z": 12}

## Step 1: Fetch curve and compute butterflies

In [3]:
ts = NYC_tz.localize(datetime.datetime.combine(AS_OF, datetime.time(17, 0)))
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
curve_handle = curve_mdp.get_pricer(request=dict(curve_name=CURVE, timestamp=ts))

contracts, prices, rates = [], [], []
for t in IMM_TENORS:
    q = IRSwapQuery(curve=CURVE, tenor=t).resolve_query(ts, pricer_or_curve=curve_handle)
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    eff = pkg[0].__dict__["kwargs"]["effective"]
    price = 100 - pkg[0].__dict__["kwargs"]["fixed_rate"]
    rate = pkg[0].__dict__["kwargs"]["fixed_rate"] * 100
    imm = ql.IMM.code(ql.Date(eff.day, eff.month, eff.year))
    contracts.append(imm)
    prices.append(price)
    rates.append(rate)

# Calendar spreads
cal_spreads = {}
for i in range(len(contracts) - 1):
    name = f"SP {contracts[i]}-{contracts[i+1]}"
    cal_spreads[name] = (prices[i] - prices[i+1]) * 10000

# 3-month butterflies
butterflies = []
for i in range(len(contracts) - 2):
    near, mid, far = contracts[i], contracts[i+1], contracts[i+2]
    bf_val = (prices[i] - 2 * prices[i+1] + prices[i+2]) * 10000
    butterflies.append({
        "name": f"BF {near}-{mid}-{far}",
        "near": near, "mid": mid, "far": far,
        "near_symbol": IMM_CODE_TO_SFR[near],
        "mid_symbol": IMM_CODE_TO_SFR[mid],
        "far_symbol": IMM_CODE_TO_SFR[far],
        "bf_bps": bf_val,
        "sp_near_bps": cal_spreads[f"SP {near}-{mid}"],
        "sp_far_bps": cal_spreads[f"SP {mid}-{far}"],
    })

print(f"Strip: {' -> '.join(contracts)}")
print(f"Rate range: {min(rates):.2f}% - {max(rates):.2f}%\n")

sp_df = pd.DataFrame({"Calendar Spread": cal_spreads.keys(), "bps": cal_spreads.values()})
bf_df = pd.DataFrame([{"Butterfly": b["name"], "bps": b["bf_bps"]} for b in butterflies])
display(sp_df.set_index("Calendar Spread").T)
display(bf_df.set_index("Butterfly").T)

Strip: M6 -> U6 -> Z6 -> H7 -> M7 -> U7 -> Z7 -> H8 -> M8 -> U8 -> Z8 -> H9
Rate range: 3.68% - 4.04%



Calendar Spread,SP M6-U6,SP U6-Z6,SP Z6-H7,SP H7-M7,SP M7-U7,SP U7-Z7,SP Z7-H8,SP H8-M8,SP M8-U8,SP U8-Z8,SP Z8-H9
bps,13.250001,12.5,9.499997,1.000002,-4.5,-6.000003,-4.499994,-0.500005,0.5,1.000005,2.249994


Butterfly,BF M6-U6-Z6,BF U6-Z6-H7,BF Z6-H7-M7,BF H7-M7-U7,BF M7-U7-Z7,BF U7-Z7-H8,BF Z7-H8-M8,BF H8-M8-U8,BF M8-U8-Z8,BF U8-Z8-H9
bps,0.750001,3.000002,8.499995,5.500002,1.500003,-1.500009,-3.999988,-1.000006,-0.500005,-1.249989


## Step 2: FOMC Meeting Overlay

In [4]:
fomc_dates_raw = get_fomc_meetings_list(as_of=AS_OF, n_plus_years=2)
fomc_dates = []
for d in fomc_dates_raw:
    if isinstance(d, datetime.datetime):
        fomc_dates.append(d.date())
    elif hasattr(d, 'date') and callable(d.date):
        fomc_dates.append(d.date())
    else:
        fomc_dates.append(d)
fomc_dates = [d for d in fomc_dates if d > AS_OF]

def imm_to_approx_dates(code):
    month = IMM_MONTH_MAP[code[0]]
    year = 2020 + int(code[1])
    start = datetime.date(year, month, 15)
    end_month = month + 3
    end_year = year
    if end_month > 12:
        end_month -= 12
        end_year += 1
    return start, datetime.date(end_year, end_month, 15)

def count_fomc_in_window(start, end, fomc_list):
    return sum(1 for d in fomc_list if start <= d < end)

for bf in butterflies:
    for leg in ["near", "mid", "far"]:
        s, e = imm_to_approx_dates(bf[leg])
        bf[f"fomc_{leg}"] = count_fomc_in_window(s, e, fomc_dates)

print(f"FOMC dates after {AS_OF}: {len(fomc_dates)}")
for d in fomc_dates[:12]:
    print(f"  {d}")

FOMC dates after 2026-05-22: 21
  2026-06-17
  2026-07-29
  2026-09-16
  2026-10-28
  2026-12-09
  2027-01-27
  2027-03-17
  2027-04-28
  2027-06-09
  2027-07-28
  2027-09-15
  2027-10-27


## Step 3: Fetch SABR Smiles & Extract BKM + BL Moments

In [5]:
stirfo_mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")

# SABR extrapolation gives a 200-point dense grid for robust BKM integration
dist = SFRImpliedDistribution(
    use_sabr_vols=True,
    sabr_extrapolation=True,
    sabr_n_strikes=200,
)

moment_cache = {}
all_symbols = sorted(set(s for bf in butterflies for s in [bf["near_symbol"], bf["mid_symbol"], bf["far_symbol"]]))

for sym in all_symbols:
    try:
        smile = stirfo_mdp.fetch_sabr_smile({
            "symbol": sym,
            "as_of": AS_OF,
            "show_tqdm": False,
        })
        snapshot = dist.extract(smile, run_bl=True, run_gm=False, run_bkm=True)
        moment_cache[sym] = {
            "bkm": snapshot.bkm_result,
            "bl": snapshot.bl_result,
            "forward_rate": smile.params.forward_rate,
            "tte": smile.params.time_to_expiry,
        }
        bkm = snapshot.bkm_result
        print(f"  {sym}: fwd={smile.params.forward_rate:.4f}% "
              f"BKM_std={bkm.std_rate*100:.1f}bps "
              f"skew={bkm.skewness_rate:+.3f} "
              f"kurt={bkm.kurtosis_rate:.2f}")
    except Exception as e:
        print(f"  {sym}: FAILED - {e}")
        moment_cache[sym] = None

  SFRH27: fwd=4.0500% BKM_std=109.6bps skew=+1.192 kurt=12.42
  SFRH28: fwd=3.9100% BKM_std=165.2bps skew=+1.219 kurt=9.88
  SFRH29: fwd=3.9400% BKM_std=197.5bps skew=+1.342 kurt=9.26
  SFRM26: fwd=3.6825% BKM_std=11.5bps skew=+1.480 kurt=66.38
  SFRM27: fwd=4.0600% BKM_std=127.7bps skew=+1.003 kurt=10.72
  SFRM28: fwd=3.9050% BKM_std=175.4bps skew=+1.285 kurt=9.66
  SFRU26: fwd=3.8150% BKM_std=46.2bps skew=+0.233 kurt=27.25
  SFRU27: FAILED - asyncio.run() cannot be called from a running event loop
  SFRU28: fwd=3.9150% BKM_std=183.7bps skew=+1.310 kurt=9.62
  SFRZ26: fwd=3.9500% BKM_std=80.5bps skew=+0.994 kurt=15.19
  SFRZ27: fwd=3.9550% BKM_std=153.5bps skew=+1.144 kurt=9.52


C:\Users\chris\AppData\Local\Temp\ipykernel_43852\2557562768.py:34: RuntimeWarning: coroutine 'BarchartFetcher.barchart_timeseries_api.<locals>.run_fetch_all' was never awaited
  moment_cache[sym] = None


  SFRZ28: FAILED - asyncio.run() cannot be called from a running event loop


## Layer 1: Variance Consistency

For each butterfly, compare the middle leg's BKM variance to the linear interpolation of wing variances.  
**Std Excess** = mid std - (near std + far std)/2  
- Positive → mid has MORE uncertainty than neighbors → convexity helps the fly → slightly rich  
- Negative → mid has LESS uncertainty → convexity hurts the fly → slightly cheap

In [6]:
for bf in butterflies:
    m_near = moment_cache.get(bf["near_symbol"])
    m_mid = moment_cache.get(bf["mid_symbol"])
    m_far = moment_cache.get(bf["far_symbol"])

    if all(m and m.get("bkm") for m in [m_near, m_mid, m_far]):
        std_n = m_near["bkm"].std_rate * 100
        std_m = m_mid["bkm"].std_rate * 100
        std_f = m_far["bkm"].std_rate * 100
        std_interp = (std_n + std_f) / 2.0
        std_excess = std_m - std_interp
        fly_abs = max(abs(bf["bf_bps"]), 0.5)

        bf["std_near"] = std_n
        bf["std_mid"] = std_m
        bf["std_far"] = std_f
        bf["std_excess"] = std_excess
        bf["var_signal"] = std_excess / fly_abs
    else:
        bf["std_near"] = bf["std_mid"] = bf["std_far"] = None
        bf["std_excess"] = bf["var_signal"] = None

## Layer 2: Probability Space

Convert calendar spreads to implied probabilities (SP/25). The butterfly/25 is the change in move probability.  
**Kink/Width** = |fly level| / mid std — measures timing conviction relative to distributional uncertainty.

In [7]:
for bf in butterflies:
    bf["prob_near"] = bf["sp_near_bps"] / 25.0
    bf["prob_far"] = bf["sp_far_bps"] / 25.0
    bf["kink_pp"] = abs(bf["bf_bps"] / 25.0) * 100

    m_mid = moment_cache.get(bf["mid_symbol"])
    if m_mid and m_mid.get("bkm"):
        mid_std = m_mid["bkm"].std_rate * 100
        bf["kink_width_ratio"] = abs(bf["bf_bps"]) / max(mid_std, 1.0)
    else:
        bf["kink_width_ratio"] = None

## Layer 3: Fragility

How sensitive is the fly to a 1% tail mass migration at its weakest leg?  
High fragility → size down regardless of signal from L1/L2.

In [8]:
TAIL_SHIFT_PCT = 0.01

for bf in butterflies:
    m_near = moment_cache.get(bf["near_symbol"])
    m_mid = moment_cache.get(bf["mid_symbol"])
    m_far = moment_cache.get(bf["far_symbol"])

    if all(m and m.get("bkm") for m in [m_near, m_mid, m_far]):
        range_n = 4.0 * m_near["bkm"].std_rate * 100
        range_m = 4.0 * m_mid["bkm"].std_rate * 100
        range_f = 4.0 * m_far["bkm"].std_rate * 100

        delta_n = TAIL_SHIFT_PCT * range_n
        delta_m = TAIL_SHIFT_PCT * range_m
        delta_f = TAIL_SHIFT_PCT * range_f

        fly_sensitivity = abs(delta_n - 2 * delta_m + delta_f)
        fly_abs = max(abs(bf["bf_bps"]), 0.5)
        bf["fragility"] = fly_sensitivity / fly_abs
        bf["weakest_leg"] = max({"near": range_n, "mid": range_m, "far": range_f},
                                key=lambda k: {"near": range_n, "mid": range_m, "far": range_f}[k])
    else:
        bf["fragility"] = None
        bf["weakest_leg"] = None

## Composite Results

In [9]:
rows = []
for bf in butterflies:
    rows.append({
        "Butterfly": bf["name"],
        "Level (bps)": round(bf["bf_bps"], 2),
        "Prob chg (pp)": round(bf["bf_bps"] / 25.0 * 100, 1),
        "Std Near": round(bf["std_near"], 1) if bf.get("std_near") else None,
        "Std Mid": round(bf["std_mid"], 1) if bf.get("std_mid") else None,
        "Std Far": round(bf["std_far"], 1) if bf.get("std_far") else None,
        "Std Excess": round(bf["std_excess"], 1) if bf.get("std_excess") is not None else None,
        "Var Signal": round(bf["var_signal"], 2) if bf.get("var_signal") is not None else None,
        "Kink/Width": round(bf["kink_width_ratio"], 3) if bf.get("kink_width_ratio") is not None else None,
        "Fragility": round(bf["fragility"], 3) if bf.get("fragility") is not None else None,
        "Weakest": bf.get("weakest_leg"),
        "FOMC": f"{bf.get('fomc_near',0)}/{bf.get('fomc_mid',0)}/{bf.get('fomc_far',0)}",
    })

results_df = pd.DataFrame(rows).set_index("Butterfly")
display(results_df)

,Level (bps),Prob chg (pp),Std Near,Std Mid,Std Far,Std Excess,Var Signal,Kink/Width,Fragility,Weakest,FOMC
Butterfly,,,,,,,,,,,
BF M6-U6-Z6,0.75,3.0,11.5,46.2,80.5,0.2,0.29,0.016,0.023,far,2/3/1
BF U6-Z6-H7,3.00,12.0,46.2,80.5,109.6,2.6,0.87,0.037,0.070,far,3/1/3
BF Z6-H7-M7,8.50,34.0,80.5,109.6,127.7,5.5,0.65,0.078,0.052,far,1/3/1
BF H7-M7-U7,5.50,22.0,NaN,NaN,NaN,NaN,NaN,0.043,NaN,None,3/1/3
BF M7-U7-Z7,1.50,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,1/3/1
BF U7-Z7-H8,-1.50,-6.0,NaN,NaN,NaN,NaN,NaN,0.010,NaN,None,3/1/2
BF Z7-H8-M8,-4.00,-16.0,153.5,165.2,175.4,0.8,0.19,0.024,0.015,far,1/2/2
BF H8-M8-U8,-1.00,-4.0,165.2,175.4,183.7,0.9,0.94,0.006,0.075,far,2/2/3
BF M8-U8-Z8,-0.50,-2.0,NaN,NaN,NaN,NaN,NaN,0.003,NaN,None,2/3/0


## Actionable Signals

In [10]:
for bf in butterflies:
    signals = []
    vs = bf.get("var_signal")
    frag = bf.get("fragility")
    kw = bf.get("kink_width_ratio")

    if vs is not None and abs(vs) > 0.3:
        direction = "RICH (sell)" if vs > 0 else "CHEAP (buy)"
        signals.append(f"L1 Variance: {direction} (std_excess/fly={vs:+.2f})")

    if kw is not None and kw > 0.15:
        signals.append(f"L2 High timing conviction (K/W={kw:.3f})")
    elif kw is not None and kw < 0.03 and abs(bf["bf_bps"]) > 1.0:
        signals.append(f"L2 Weak kink vs uncertainty (K/W={kw:.3f})")

    if frag is not None and frag > 0.5:
        signals.append(f"L3 HIGH fragility ({frag:.3f}) -> size down")
    elif frag is not None and frag < 0.15:
        signals.append(f"L3 LOW fragility ({frag:.3f}) -> full size OK")

    fomc_asym = abs(bf.get("fomc_near", 0) - bf.get("fomc_far", 0))
    if fomc_asym >= 1:
        signals.append(f"FOMC: asymmetric ({bf['fomc_near']}/{bf['fomc_mid']}/{bf['fomc_far']})")

    if signals:
        print(f"\n{bf['name']} ({bf['bf_bps']:+.2f} bps):")
        for s in signals:
            print(f"  - {s}")


BF M6-U6-Z6 (+0.75 bps):
  - L3 LOW fragility (0.023) -> full size OK
  - FOMC: asymmetric (2/3/1)

BF U6-Z6-H7 (+3.00 bps):
  - L1 Variance: RICH (sell) (std_excess/fly=+0.87)
  - L3 LOW fragility (0.070) -> full size OK

BF Z6-H7-M7 (+8.50 bps):
  - L1 Variance: RICH (sell) (std_excess/fly=+0.65)
  - L3 LOW fragility (0.052) -> full size OK

BF U7-Z7-H8 (-1.50 bps):
  - L2 Weak kink vs uncertainty (K/W=0.010)
  - FOMC: asymmetric (3/1/2)

BF Z7-H8-M8 (-4.00 bps):
  - L2 Weak kink vs uncertainty (K/W=0.024)
  - L3 LOW fragility (0.015) -> full size OK
  - FOMC: asymmetric (1/2/2)

BF H8-M8-U8 (-1.00 bps):
  - L1 Variance: RICH (sell) (std_excess/fly=+0.94)
  - L2 Weak kink vs uncertainty (K/W=0.006)
  - L3 LOW fragility (0.075) -> full size OK
  - FOMC: asymmetric (2/2/3)

BF M8-U8-Z8 (-0.50 bps):
  - FOMC: asymmetric (2/3/0)

BF U8-Z8-H9 (-1.25 bps):
  - FOMC: asymmetric (3/0/0)
